In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append('../../scripts')

In [3]:
import numpy as np
import scanpy as sc
import os
DATA_ROOT = '/data2/a330d' #os.environ.get("DATA_ROOT", ".")
import matplotlib.pyplot as plt
import decoupler as dc
import scipy.sparse as sp
import pandas as pd

from scipy.stats import pearsonr, spearmanr

from utils import set_seed
from train_loo import preprocess_crc, preprocess_merfish, _load_model, preprocess_spatial_features
from counterfactual_analysis import compute_rmse, compute_edistance, mixing_index, get_lfc, precision, direction_match, compute_mse_lfc, _to_dense
from configs.adata_crc_config import ADATA_ARGS as ADATA_ARGS_CRC
from configs.adata_merfish_config import ADATA_ARGS as ADATA_ARGS_MERFISH
from configs.cellina_config import MODEL_ARGS as CELLINA_MODEL_ARGS, TRAIN_ARGS as CELLINA_TRAIN_ARGS, PLAN_KWARGS as CELLINA_PLAN_KWARGS, DO_COUNTERFACTUAL as CELLINA_DO_COUNTERFACTUAL


/data/a330d/miniforge3/envs/cellina-graph/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import cellina

In [5]:
set_seed(0)

In [6]:
DATASET_NAME = "crc"  # or "merfish"
MODEL_ROOT = os.path.join(DATA_ROOT, "data/ood/trained/loo_patients")

In [7]:
CRC_PATHS = [
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_210.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_221.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_231.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_232.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_242.h5ad"),
    #os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_120.h5ad"),
]

CRC_HOLDOUTS = [
    "Endothelial",
    "Epithelial",
    "Fibroblast",
    "Myeloid",
    "T_cell",
]

MERFISH_PATHS = [
    os.path.join(DATA_ROOT, "datasets/MERFISH_mouse_brain/C57BL6J-2.036.h5ad"),    
    os.path.join(DATA_ROOT, "datasets/MERFISH_mouse_brain/C57BL6J-2.039.h5ad"),
    os.path.join(DATA_ROOT, "datasets/MERFISH_mouse_brain/C57BL6J-2.041.h5ad"),
]

MERFISH_HOLDOUTS = [
    'glutamatergic neuron',
    'oligodendrocyte',
    'astrocyte',
    'GABAergic neuron',
    'endothelial cell',
]

PATHS = CRC_PATHS if DATASET_NAME == "crc" else MERFISH_PATHS
HOLDOUT_CELLTYPES = CRC_HOLDOUTS if DATASET_NAME == "crc" else MERFISH_HOLDOUTS
DATA_ARGS = ADATA_ARGS_CRC if DATASET_NAME == "crc" else ADATA_ARGS_MERFISH
COUNTS_PER_K = 1e4

In [8]:
n_top_genes = DATA_ARGS.get('n_top_genes')
labels_key = DATA_ARGS.get('labels_key')
domains_key = DATA_ARGS.get('domains_key')
batch_key = DATA_ARGS.get('batch_key')
control_domain = DATA_ARGS.get('control_domains')[0]
holdout_domains = DATA_ARGS.get('holdout_domains')
n_neighbors = DATA_ARGS.get('n_neighbors')
batch_size = 2048
library_size = 'latent'
n_deg = 50

In [9]:
adata = sc.read_h5ad('/data2/a330d/datasets/crc/processed/crc_cosmx_wt.h5ad')

In [10]:
adata = adata[adata.obs[batch_key]!=110].copy()

In [11]:
import gc
gc.collect()

2383

In [12]:
adata

AnnData object with n_obs × n_vars = 2360478 × 2000
    obs: 'fov', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'Width', 'Height', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD68_CK8_18', 'Max.CD68_CK8_18', 'Mean.CD298_B2M', 'Max.CD298_B2M', 'Mean.CD45', 'Max.CD45', 'Mean.DAPI', 'Max.DAPI', 'cell_id', 'version', 'dualfiles', 'Run_name', 'Run_Tissue_name', 'ISH.concentration', 'Dash', 'tissue', 'Panel', 'assay_type', 'slide_ID', 'CenterX_global_px', 'CenterY_global_px', 'cell_ID', 'unassignedTranscripts', 'median_RNA', 'RNA_quantile_0.75', 'RNA_quantile_0.8', 'RNA_quantile_0.85', 'RNA_quantile_0.9', 'RNA_quantile_0.95', 'RNA_quantile_0.99', 'nCount_RNA', 'nFeature_RNA', 'median_negprobes', 'negprobes_quantile_0.75', 'negprobes_quantile_0.8', 'negprobes_quantile_0.85', 'negprobes_quantile_0.9', 'negprobes_quantile_0.95', 'negprobes_quantile_0.99', 'nCount_negprobes', 'nFeature_negprobes', 'median_falsecode', 'falsecode_quantile_0.75', 'falsecode_quantile_0.8', 'falsecode_quantil

In [ ]:
# 120, 210, 242
HOLDOUT_SIDS = [210, 242]

In [14]:
def split_indices(
    adata,
    holdout_slide,
    batch_key,
    seed=0,
):
    if holdout_slide not in adata.obs[batch_key].unique():
        raise ValueError(f"holdout_slide '{holdout_slide}' not found in adata.obs['{batch_key}'] values")

    is_holdout_slide = adata.obs[batch_key] == holdout_slide
    test_mask = is_holdout_slide

    all_idx = np.arange(adata.n_obs)
    test_idx = np.where(test_mask.values)[0]
    trainval_idx = np.setdiff1d(all_idx, test_idx)

    rng = np.random.default_rng(seed)
    n_trainval = trainval_idx.shape[0]
    n_val = max(1, int(0.1 * n_trainval))
    val_idx_rel = rng.choice(np.arange(n_trainval), size=n_val, replace=False)
    val_idx = trainval_idx[val_idx_rel]
    train_idx = np.setdiff1d(trainval_idx, val_idx)

    # annotate is_holdout in adata.obs
    adata.obs['is_holdout'] = False
    if len(test_idx) > 0:
        adata.obs.iloc[test_idx, adata.obs.columns.get_loc('is_holdout')] = True

    return train_idx, val_idx, test_idx

In [15]:
model_names = ['cellina']
results = []
for slide_id in HOLDOUT_SIDS:
    for model_name in model_names:
        model_class = 'cellina' if model_name == 'cellina' else 'cellina_graph'
        # 50 times * in print
        print(f"{'='*50} Holout slide: {slide_id} {'='*50}")
        # create splits
        train_idx, val_idx, test_idx = split_indices(adata,
                                                    holdout_slide=slide_id,
                                                    batch_key=batch_key,
                                                    seed=0)

        splits = (train_idx, val_idx, test_idx)
        save_dir = os.path.join(MODEL_ROOT, str(slide_id), model_name)

        # Train model
        if model_class == 'cellina':
            from cellina import Cellina
            model_args = CELLINA_MODEL_ARGS.copy()
            train_args = CELLINA_TRAIN_ARGS.copy()
            plan_kwargs = CELLINA_PLAN_KWARGS.copy()
            
            Cellina.setup_anndata(adata, 
                                    batch_key=batch_key, 
                                    labels_key=labels_key, 
                                    domains_key=domains_key, 
                                    spatial_obsm_key='spatial_x', 
                                    layer='counts')
            model = Cellina(adata, **model_args)

            # Add split info
            train_args['datasplitter_kwargs'] = {
                    "external_indexing": [splits[0], splits[1], splits[2]],
                    }
            if plan_kwargs is not None:
                model.train(**train_args, plan_kwargs=plan_kwargs)
            else:
                model.train(**train_args)
            model.save(save_dir, overwrite=True)

================================================== Holout slide: 210 ==================================================
INFO     Generating sequential column names                                                                        
INFO     cellina: The Cellina model has been initialized with adversarial domain forgetting                        


INFO: GPU available: True (cuda), used: True
2026-07-24 15:01:13 | [INFO] GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
2026-07-24 15:01:13 | [INFO] TPU available: False, using: 0 TPU cores
INFO: You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
2026-07-24 15:01:13 | [INFO] You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
INFO: L

Epoch 10/100:   9%|▉         | 9/100 [03:54<40:54, 26.97s/it, v_num=1, train_loss=-50.2]

INFO: 
Detected KeyboardInterrupt, attempting graceful shutdown ...
2026-07-24 15:05:12 | [INFO] 
Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/data/a330d/miniforge3/envs/cellina-graph/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
model_names = ['cellina']
results = []
for slide_id in HOLDOUT_SIDS:
    for model_name in model_names:
        model_class = 'cellina' if model_name == 'cellina' else 'cellina_graph'
        # 50 times * in print
        print(f"{'='*50} Holout slide: {slide_id} {'='*50}")
        # create splits
        train_idx, val_idx, test_idx = split_indices(adata,
                                                    holdout_slide=slide_id,
                                                    batch_key=batch_key,
                                                    seed=0)

        splits = (train_idx, val_idx, test_idx)
        save_dir = os.path.join(MODEL_ROOT, str(slide_id), model_name)
        # Load model and eval
        try:
            model = _load_model(save_dir,
                                model_class=model_class,
                                adata=adata,
                                splits=splits)
        except Exception as e:
            print(f"Failed to load model from {save_dir} with error: {e}")
            continue
        for holdout_celltype in HOLDOUT_CELLTYPES:
            print(f"{'='*50} Holdout celltype: {holdout_celltype}{'='*50}")
            adata_holdout = adata[adata.obs[batch_key] == slide_id]
            is_control_region = adata_holdout.obs[domains_key]==(control_domain)
            is_holdout_ct = adata_holdout.obs[labels_key].astype(str) == holdout_celltype
            mask_control = is_control_region & is_holdout_ct
            idx_control = np.where(mask_control.values)[0]    
            
            for hd in holdout_domains:
                    is_holdout_region = adata_holdout.obs[domains_key].astype(str) == hd
                    mask_ct_target = is_holdout_ct & is_holdout_region
                    idx_target = np.where(mask_ct_target.values)[0]

                    # "neighbour_indices" are indices of the neighbors of idx_target cells
                    step_size_px = 0.12028 if DATASET_NAME == 'crc' else 0.109
                    preprocess_spatial_features(adata_holdout, step_size_px=step_size_px, n_neighbors=200, test_indices=idx_target)
                    conn = adata_holdout.obsp["spatial_connectivities_orig"]
                    sub_conn = conn[idx_target]                # rows for target cells
                    neighbor_indices = sub_conn.nonzero()[1]   # all neighbors at once
                    neighbor_indices = np.unique(neighbor_indices)
                    # keep only non-holdout-ct neighbors
                    neighbor_indices = neighbor_indices[~is_holdout_ct.values[neighbor_indices]]

                    args_gex = {
                        "adata": adata_holdout,
                        "indices": idx_control,
                        "batch_size": batch_size,
                        "seed": 0,
                        "neighbour_indices": neighbor_indices
                    }
                    if model_class.lower() == 'cellina_graph':
                        args_gex["n_neighbors_per_seed"] = 50
                    else:
                        args_gex['precomputed'] = True
                    
                    cf_counts = model.get_counterfactual_expression(**args_gex)
                    
                    # Compute stats
                    control = adata_holdout.layers['counts'][mask_control.values, :]
                    target = adata_holdout.layers['counts'][mask_ct_target.values, :]
                    control, target = _to_dense(control), _to_dense(target)
                    counterfactual = cf_counts

                    gt_lfc, cf_lfc, deg = get_lfc(control=control, target=target, counterfactual=counterfactual, n_deg=n_deg)

                    spear, _ = spearmanr(gt_lfc[deg], cf_lfc[deg])
                    pear, _ = pearsonr(gt_lfc[deg], cf_lfc[deg])
                    prec = precision(gt_lfc, cf_lfc, k=n_deg, use_abs=True)
                    dir_match = direction_match(gt_lfc, cf_lfc, k=n_deg, normalize="intersection")
                    dir_match_k = direction_match(gt_lfc, cf_lfc, k=n_deg, normalize="k")
                    dir_match_gt = direction_match(gt_lfc, cf_lfc, k=n_deg, normalize="gt_topk")
                    mix_idx = mixing_index(observed=target, predicted=counterfactual, library_size=COUNTS_PER_K)
                    edist_global = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=COUNTS_PER_K)
                    edist_local = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=COUNTS_PER_K, local=True)
                    edist_pca_log = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=COUNTS_PER_K, local=True, use_pca=True)
                    edist_pca = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=COUNTS_PER_K, local=True, use_pca=True, log1p=False)
                    rmse = compute_rmse(observed=target, predicted=counterfactual, deg=deg, library_size=COUNTS_PER_K)
                    mse_lfc = compute_mse_lfc(gt_vec=gt_lfc, cf_vec=cf_lfc, deg=deg)

                    results.append(
                            dict(
                            dataset_name=DATASET_NAME,
                            sid=slide_id,
                            control_domain=control_domain,
                            target_domain=hd,
                            n_deg=n_deg,
                            model_name=model_name,
                            holdout_celltype=holdout_celltype,
                            spearman=spear,
                            pearson=pear,
                            precision=prec,
                            direction_match=dir_match,
                            direction_match_k=dir_match_k,
                            direction_match_gt=dir_match_gt,
                            mixing_index=mix_idx,
                            edistance_global=edist_global,
                            edistance_local=edist_local,
                            edistance_pca_log=edist_pca_log,
                            edistance_pca=edist_pca,
                            rmse=rmse,
                            mse_lfc=mse_lfc,
                            )
                    )
            gc.collect()

In [17]:
# Append to existing csv if exists, otherwise create new csv
results_csv_name = f'../../results/loo_cellina_{DATASET_NAME}_DEG_{n_deg}_patients.csv'

df_results = pd.DataFrame(results)
if os.path.exists(results_csv_name):
    df_results.to_csv(f"{results_csv_name}", index=False, mode='a', header=False)
else:
    df_results.to_csv(f"{results_csv_name}", index=False)

In [18]:
df_results

,dataset_name,sid,control_domain,target_domain,n_deg,model_name,holdout_celltype,spearman,pearson,precision,direction_match,direction_match_k,direction_match_gt,mixing_index,edistance_global,edistance_local,edistance_pca_log,edistance_pca,rmse,mse_lfc
0,crc,120,REF,CRC,50,cellina,Endothelial,0.650420,0.951726,0.44,1.000,0.44,1.00,0.578569,67.360249,67.419335,4.824498,143.067209,5589.729175,0.403701
1,crc,120,REF,CRC,50,cellina,Epithelial,0.534118,0.591247,0.16,0.875,0.14,0.82,0.906495,50.541346,46.670808,7.905737,242.447326,625586.979438,7.686411
2,crc,120,REF,CRC,50,cellina,Fibroblast,0.537575,0.550874,0.12,1.000,0.12,0.78,0.699028,61.740529,58.951569,6.006607,237.932807,178721.052434,1.343545
3,crc,120,REF,CRC,50,cellina,Myeloid,0.686819,0.759237,0.12,1.000,0.12,0.90,0.833297,61.748052,60.963712,4.777579,136.739822,47047.720167,0.446454
4,crc,120,REF,CRC,50,cellina,T_cell,0.735510,0.859796,0.08,1.000,0.08,0.96,0.852620,72.455713,74.620666,5.184029,103.424030,22639.498999,0.328682
